# Analyse von Boilerplate-Phrasen, Navigations-Bias und Korpus-Bereinigung
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte/Einfache Sprache (LS)

In dieser Untersuchung analysieren wir den Korpus im direkten **Vorher-Nachher-Vergleich**:
1. **Rohdaten ()**: Unbereinigte Web-Scraping-Daten mit systematischen Web-Floskeln (*„Mehr Informationen...“*, *„Wo finde ich weitere Infos?“*, *„Klicken Sie hier“*), Prüfstellen-Signaturen (*„Universität Hildesheim“*, *„Stefan Albers“*) und Hunderten redundanten URL-Klonen.
2. **Bereinigter Korpus ()**: Vollständig normalisiert, von Web-Boilerplate und Navigations-Frageketten bereinigt, dedupliziert und nach Längenverhältnis gefiltert.

Wir quantifizieren hier exakt den Einfluss der Bereinigung auf die **Artikelpaare**, **Wort- und Tokenzahlen**, **Längenverhältnisse** und die **Eliminierung des Boilerplate-Bias**.

In [ ]:
import os
import sys
import json
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Arbeitsverzeichnis auf Projekt-Root setzen
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")

print("Aktuelles Arbeitsverzeichnis:", os.getcwd())

# Plot-Stil
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 1. Bereinigungs- und Filterlogik aus 
Wir binden exakt dieselben Funktionen ein, die im zentralen Vorverarbeitungsskript  für die Datenpipeline definiert sind.

In [ ]:
def clean_navigation_boilerplate(text):
    # 1. Question headers asking where to find more info / how to navigate
    text = re.sub(r"Wo (finde|bekomme) ich (noch |weitere |mehr )?(Informationen|Infos).*?\?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Sie möchten (noch |weitere |mehr )?(Informationen|Infos).*?\?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Sie wollen noch mehr (über|zu).*?lesen\??", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Sie interessieren sich für.*?\?(\s*Dann klicken Sie.*?\.)?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Welche Frage zu.*?haben Sie\?\s*Unser Tool durchsucht unsere Artikel.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Ihre Frage wird nicht gespeichert\.\s*(Augen|[A-Za-z]+)?", "", text, flags=re.IGNORECASE)
    
    # 2. Sentences referring to more info / links on this page or external sites
    text = re.sub(r"Hier (erfahren|lesen|bekommen|finden) Sie (mehr|alles|weitere|etwas).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(Erfahren|Lesen) Sie mehr (über|zu|zum Thema).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Mehr (Informationen|Infos) (über|zu|in Alltagssprache|im Internet).*?(auf dieser Seite|hier|finden Sie|erfahren Sie|lesen Sie|unter).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Mehr Infos (zu|zur|zum)\s+[\w\-]+:?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Hier finden Sie (weitere )?Infos.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Hier kommen Sie (zum|zur|zu).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Sprechen Sie uns an!?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Auf dieser Seite finden Sie (viele |mehr |weitere )?Informationen zum Thema:.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Weitere (Informationen|Infos) (in Alltagssprache|zum Thema|über).*?(\.|$", "", text, flags=re.IGNORECASE)
    
    # 3. Link lists & click instructions
    text = re.sub(r"(\((in )?Alltagssprache\))", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Einen Link (für|zu).*?(unten|hier).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Links?\s+(Link\s+)?(zum|zur|zu|unter).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"(Wenn Sie.*?online lesen möchten,\s*)?([kK]licken|[tT]ippen) Sie (bitte )?(auf|hier).*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Dann finden Sie in dieser Tabelle weitere Informationen.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Achtung:\s*Dieser Link führt.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Die Informationen sind dann nicht mehr in (Einfacher|Leichter) Sprache.*?(\.|$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Hier geht es zu.*?(\.|$", "", text, flags=re.IGNORECASE)

    # 4. Clean up residual double spaces and empty punctuation
    text = re.sub(r"\s+([?.!,;])", r"", text)
    return re.sub(r"\s+", " ", text).strip()


## 2. Daten laden: Vorher (Rohdaten) vs. Nachher (Bereinigt)

In [ ]:
raw_files = sorted(glob.glob("data/corpus/2_raw_scraped/*.json"))
if not raw_files:
    raw_files = sorted(glob.glob("data/corpus/4_normalized_clean/*.json"))

raw_data = []
clean_data = []

for file_path in tqdm(raw_files, desc="Lade und verarbeite Korpusdaten"):
    source_name = os.path.basename(file_path).replace("_articles.json", "")
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    pairs = data.get("pairs", [])
    seen_keys = set()
    
    for pair in pairs:
        ls_raw = pair.get("ls_text", "")
        as_raw = pair.get("as_text", "")
        
        if not ls_raw or not as_raw:
            continue
            
        # Rohdaten erfassen
        raw_data.append({
            "source": source_name,
            "ls_text": ls_raw,
            "as_text": as_raw,
            "ls_words": len(ls_raw.split()),
            "as_words": len(as_raw.split()),
            "ls_chars": len(ls_raw),
            "as_chars": len(as_raw),
            "word_ratio": len(ls_raw.split()) / max(1, len(as_raw.split()))
        })
        
        # Bereinigungsschritte anwenden
        ls_c = normalize_mediopunkt(ls_raw)
        as_c = normalize_mediopunkt(as_raw) if as_raw else ""
        
        if source_name == "brandeins": ls_c = clean_brandeins(ls_c)
        elif source_name == "mdr": ls_c = clean_mdr(ls_c)
        elif source_name == "taz": ls_c = clean_taz(ls_c); ls_c = clean_taz_hamburg_credits(ls_c)
        elif source_name == "hamburg": ls_c = clean_taz_hamburg_credits(ls_c)
        elif source_name == "apotheken": ls_c = clean_apotheken(ls_c)
        elif source_name == "hannover": ls_c = clean_hannover(ls_c)
        elif source_name == "stuttgart": ls_c = clean_stuttgart(ls_c)
        elif source_name == "koeln": ls_c = clean_stuttgart_koeln(ls_c)
        
        if source_name == "hannover" and as_c:
            as_c = clean_hannover(as_c)
            
        ls_c = clean_navigation_boilerplate(ls_c)
        if as_c: as_c = clean_navigation_boilerplate(as_c)
        
        ls_c = re.sub(r"\s+", " ", ls_c).strip()
        as_c = re.sub(r"\s+", " ", as_c).strip()
        
        # Längenfilter (Ratio & Mindestwörter)
        if not is_valid_length_ratio(ls_c, as_c):
            continue
            
        # Exakte Deduplizierung (Entfernt die 513 URL-Klone)
        key = (ls_c, as_c)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        
        clean_data.append({
            "source": source_name,
            "ls_text": ls_c,
            "as_text": as_c,
            "ls_words": len(ls_c.split()),
            "as_words": len(as_c.split()),
            "ls_chars": len(ls_c),
            "as_chars": len(as_c),
            "word_ratio": len(ls_c.split()) / max(1, len(as_c.split()))
        })

df_raw = pd.DataFrame(raw_data)
df_clean = pd.DataFrame(clean_data)

print(f"Geladene Rohpaare (Vorher):   {len(df_raw)}")
print(f"Bereinigte Paare (Nachher):   {len(df_clean)}")

## 3. Makro-Vergleich: Artikelpaare, Wörter & Tokens (Vorher vs. Nachher)

In [ ]:
# Zusammenfassung der Artikelpaare pro Quelle
summary_pairs = pd.DataFrame({
    "Artikelpaare (Vorher)": df_raw.groupby("source").size(),
    "Artikelpaare (Nachher)": df_clean.groupby("source").size(),
}).fillna(0).astype(int)

summary_pairs["Entfernte Klone/Ausreißer"] = summary_pairs["Artikelpaare (Vorher)"] - summary_pairs["Artikelpaare (Nachher)"]
summary_pairs["Beibehaltungsrate (%)"] = (summary_pairs["Artikelpaare (Nachher)"] / summary_pairs["Artikelpaare (Vorher)"] * 100).round(1)

# Wort- und Token-Statistiken
word_stats = pd.DataFrame({
    "LS Wörter (Vorher)": df_raw.groupby("source")["ls_words"].sum(),
    "LS Wörter (Nachher)": df_clean.groupby("source")["ls_words"].sum(),
    "AS Wörter (Vorher)": df_raw.groupby("source")["as_words"].sum(),
    "AS Wörter (Nachher)": df_clean.groupby("source")["as_words"].sum(),
}).fillna(0).astype(int)

comparison_table = pd.concat([summary_pairs, word_stats], axis=1)

# Gesamtzeile hinzufügen
totals = pd.Series({
    "Artikelpaare (Vorher)": df_raw.shape[0],
    "Artikelpaare (Nachher)": df_clean.shape[0],
    "Entfernte Klone/Ausreißer": df_raw.shape[0] - df_clean.shape[0],
    "Beibehaltungsrate (%)": round(df_clean.shape[0] / df_raw.shape[0] * 100, 1),
    "LS Wörter (Vorher)": df_raw["ls_words"].sum(),
    "LS Wörter (Nachher)": df_clean["ls_words"].sum(),
    "AS Wörter (Vorher)": df_raw["as_words"].sum(),
    "AS Wörter (Nachher)": df_clean["as_words"].sum()
}, name="GESAMT")

full_summary = pd.concat([comparison_table, pd.DataFrame([totals])])
full_summary

### Visualisierung: Artikelpaare und Textvolumen vor und nach Bereinigung

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Artikelpaare pro Quelle
plot_df = summary_pairs[["Artikelpaare (Vorher)", "Artikelpaare (Nachher)"]].reset_index()
plot_df_melted = pd.melt(plot_df, id_vars=["source"], value_vars=["Artikelpaare (Vorher)", "Artikelpaare (Nachher)"], var_name="Status", value_name="Anzahl")

sns.barplot(data=plot_df_melted, x="source", y="Anzahl", hue="Status", palette=["#e74c3c", "#2ecc71"], ax=axes[0])
axes[0].set_title("Artikelpaare pro Quelle (Vorher vs. Nachher)", fontsize=13, fontweight="bold")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].set_ylabel("Anzahl Artikelpaare")

# Plot 2: Wortlängenverhältnis (LS / AS)
ratio_df = pd.DataFrame({
    "Ratio": list(df_raw["word_ratio"]) + list(df_clean["word_ratio"]),
    "Status": ["Vorher (Rohdaten)"] * len(df_raw) + ["Nachher (Bereinigt)"] * len(df_clean)
})

sns.boxplot(data=ratio_df, x="Status", y="Ratio", palette=["#e74c3c", "#2ecc71"], showfliers=False, ax=axes[1])
axes[1].set_title("Längenverhältnis LS/AS (Ausreißer-Bereinigung)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Verhältnis (Wörter LS / Wörter AS)")

plt.tight_layout()
plt.show()

## 4. Boilerplate-Bias Kategorien & Erkennung
Wir kategorisieren alle bekannten Web- und Navigationsfloskeln in fünf Hauptgruppen:

In [ ]:
boilerplate_categories = {
    "1. Mehr Informationen/Links": ["mehr informationen", "mehr infos", "weitere informationen", "auf dieser seite", "im internet unter", "hier finden sie", "hier gibt es"],
    "2. Frageketten & Nav": ["wo finde ich", "wo bekomme ich", "wie kann ich", "sie möchten", "sie wollen noch mehr", "sie interessieren sich für", "unser tool durchsucht"],
    "3. Achtung / Link-Warnung": ["dieser link führt", "aus unserem einfache-sprache-angebot", "nicht mehr in einfacher sprache", "achtung:", "wichtig:"],
    "4. Prüfstellen/Credits": ["forschungsstelle leichte sprache", "universität hildesheim", "stefan albers", "atelier fleetinsel", "european easy-to-read logo", "inclusion europe", "geprüft von", "übersetzt von", "institut für leichte sprache", "übertragung in leichte sprache"],
    "5. Regionale & Bild-Artefakte": ["region hannover", "icon für die mobilversion", "fotografiert von", "infobus", "das team vom"]
}

# Berechne Vorkommen in Vorher (df_raw)
for cat, patterns in boilerplate_categories.items():
    df_raw[cat] = df_raw["ls_text"].apply(lambda x: any(re.search(re.escape(p), x, re.IGNORECASE) for p in patterns))
df_raw["has_any_boilerplate"] = df_raw[list(boilerplate_categories.keys())].any(axis=1)

# Berechne Vorkommen in Nachher (df_clean)
for cat, patterns in boilerplate_categories.items():
    df_clean[cat] = df_clean["ls_text"].apply(lambda x: any(re.search(re.escape(p), x, re.IGNORECASE) for p in patterns))
df_clean["has_any_boilerplate"] = df_clean[list(boilerplate_categories.keys())].any(axis=1)

print(f"Anteil Artikel mit Boilerplate VORHER:  {df_raw['has_any_boilerplate'].mean():.1%}")
print(f"Anteil Artikel mit Boilerplate NACHHER: {df_clean['has_any_boilerplate'].mean():.1%}")

### Heatmap-Vergleich: Boilerplate-Verteilung pro Quelle (Vorher vs. Nachher)

In [ ]:
heat_raw = df_raw.groupby("source")[list(boilerplate_categories.keys()) + ["has_any_boilerplate"]].mean() * 100
heat_clean = df_clean.groupby("source")[list(boilerplate_categories.keys()) + ["has_any_boilerplate"]].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(heat_raw, annot=True, fmt=".1f", cmap="Reds", cbar=False, ax=axes[0], vmin=0, vmax=100)
axes[0].set_title("Boilerplate-Prävalenz VORHER (% der Artikel)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Quelle")

sns.heatmap(heat_clean, annot=True, fmt=".1f", cmap="Greens", cbar=True, ax=axes[1], vmin=0, vmax=100)
axes[1].set_title("Boilerplate-Prävalenz NACHHER (% der Artikel)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 5. Qualitative Fallstudien (Vorher / Nachher Text-Deltas)
Hier sehen wir konkrete Textbeispiele aus den wichtigsten Quellen (Apotheken, Hannover, Köln/Stuttgart), die zeigen, wie Navigationsfragmente restlos entfernt wurden, während der inhaltliche Kern vollständig erhalten bleibt.

In [ ]:
sample_sources = ["apotheken", "hannover", "stuttgart", "koeln"]

for src in sample_sources:
    raw_sub = df_raw[df_raw["source"] == src]
    clean_sub = df_clean[df_clean["source"] == src]
    
    if not raw_sub.empty and not clean_sub.empty:
        print("=" * 80)
        print(f"BEISPIEL AUS QUELLE: {src.upper()}")
        print("=" * 80)
        
        r_text = raw_sub.iloc[0]["ls_text"]
        c_text = clean_sub.iloc[0]["ls_text"]
        
        print("--- [VORHER (Letzte 300 Zeichen)] ---")
        print("... " + r_text[-300:])
        print("
--- [NACHHER (Letzte 300 Zeichen)] ---")
        print("... " + c_text[-300:])
        print("
")